## Exercise 1: Exploratory Data Analysis

### Instructions:
1. Load the data from CSV files
2. Remove target column from the training data
3. Split the data into train/test split
4. Understand the data

First, we need to load the dataset. Assuming the `heart.csv` file is available in your Colab environment or session storage. If not, you might need to upload it or download it from a source like Kaggle. For example, if you download it from Kaggle, you can upload it directly to your Colab session. Or, if it's hosted online, you can use `pd.read_csv` directly with the URL.

In [ ]:
import pandas as pd

# Load the dataset
try:
    df = pd.read_csv('heart.csv')
    print("Dataset loaded successfully.")
    display(df.head())
except FileNotFoundError:
    print("Error: 'heart.csv' not found. Please upload the dataset or provide the correct path.")
    # You can add code here to download the file if a URL is known, e.g.:
    # df = pd.read_csv('https://raw.githubusercontent.com/username/repo/main/heart.csv')
    # display(df.head())

### Understand the data

Let's inspect the data types, non-null values, and summary statistics to get a better understanding of the dataset.

In [ ]:
# Display basic information about the DataFrame
print(df.info())

print('\n--- Descriptive Statistics ---\n')
# Display descriptive statistics for numerical columns
display(df.describe())

### Remove target column and split the data

The target column, typically named 'target', needs to be separated from the features (X) and then the data will be split into training and testing sets.

In [ ]:
from sklearn.model_selection import train_test_split

# Assuming 'target' is the column to predict
if 'target' in df.columns:
    X = df.drop('target', axis=1) # Features
    y = df['target']             # Target variable

    # Split the data into training and testing sets
    # Using a common split ratio of 80% train, 20% test, with a random_state for reproducibility
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"Original dataset shape: {df.shape}")
    print(f"Features (X) shape: {X.shape}")
    print(f"Target (y) shape: {y.shape}")
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")
else:
    print("Error: 'target' column not found in the dataset. Please verify the target column name.")

## Exercise 2: Logistic Regression without Grid Search

### Instructions:
Use the dataset to build a logistic regression model without using grid search. Split the data into training and testing sets, then train a logistic regression model and evaluate its performance on the test set.

We will now train a Logistic Regression model using the `X_train` and `y_train` data, and then evaluate it on `X_test` and `y_test`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize the Logistic Regression model
# Using a default solver and random_state for reproducibility
log_reg_model = LogisticRegression(solver='liblinear', random_state=42)

# Train the model using the training data
log_reg_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

### Evaluate Model Performance

Now, let's make predictions on the test set and evaluate the model's performance using common classification metrics such as accuracy, classification report, and confusion matrix.

In [ ]:
# Make predictions on the test set
y_pred_log_reg = log_reg_model.predict(X_test)

# Calculate accuracy
accuracy_log_reg = accuracy_score(y_test, y_pred_log_reg)
print(f"Accuracy of Logistic Regression Model: {accuracy_log_reg:.4f}")

# Display classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_log_reg))

# Display confusion matrix
print("\nConfusion Matrix:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_log_reg),
                     index=['Actual Negative', 'Actual Positive'],
                     columns=['Predicted Negative', 'Predicted Positive']))

## Exercise 3: Logistic Regression with Grid Search

### Instructions:
Build a logistic regression model using the dataset, but this time, use `GridSearchCV` to optimize the hyperparameters such as `C` and `penalty`.

We will now use `GridSearchCV` to find the optimal hyperparameters for our Logistic Regression model. This involves defining a parameter grid and letting `GridSearchCV` explore the best combination using cross-validation.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for Logistic Regression
# 'C' is the inverse of regularization strength; smaller values specify stronger regularization.
# 'penalty' specifies the norm (L1 or L2) used in the penalization.
param_grid_log_reg = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2']
}

# Initialize the Logistic Regression model again
# Note: 'liblinear' solver supports both 'l1' and 'l2' penalties.
logistic_model_gs = LogisticRegression(solver='liblinear', random_state=42)

# Initialize GridSearchCV
# cv=5 means 5-fold cross-validation
# scoring='accuracy' means the best model will be chosen based on accuracy
# n_jobs=-1 uses all available processors
grid_search_log_reg = GridSearchCV(logistic_model_gs, param_grid_log_reg, cv=5, scoring='accuracy', n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search_log_reg.fit(X_train, y_train)

print("GridSearchCV for Logistic Regression completed.")

### Best Parameters and Evaluation

Let's examine the best parameters found by `GridSearchCV` and then evaluate the performance of the best estimator on the test set.

In [ ]:
# Print the best parameters found by GridSearchCV
print("Best parameters for Logistic Regression:", grid_search_log_reg.best_params_)

# Print the best cross-validation score
print("Best cross-validation accuracy:", grid_search_log_reg.best_score_)

# Get the best estimator (model) from GridSearchCV
best_log_reg_model = grid_search_log_reg.best_estimator_

# Make predictions on the test set using the best model
y_pred_best_log_reg = best_log_reg_model.predict(X_test)

# Calculate accuracy of the best model
accuracy_best_log_reg = accuracy_score(y_test, y_pred_best_log_reg)
print(f"\nAccuracy of Best Logistic Regression Model (with Grid Search): {accuracy_best_log_reg:.4f}")

# Display classification report for the best model
print("\nClassification Report for Best Logistic Regression Model:")
print(classification_report(y_test, y_pred_best_log_reg))

# Display confusion matrix for the best model
print("\nConfusion Matrix for Best Logistic Regression Model:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_best_log_reg),
                     index=['Actual Negative', 'Actual Positive'],
                     columns=['Predicted Negative', 'Predicted Positive']))

## Exercise 4: SVM without Grid Search

### Instructions:
Train a Support Vector Machine (SVM) classifier on the dataset without using grid search. Choose an appropriate kernel and set the hyperparameters manually.

We'll use a basic SVM with a radial basis function (RBF) kernel as a common starting point.

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize the SVM classifier with a chosen kernel and hyperparameters
# For example, using a radial basis function (RBF) kernel with default C=1.0 and gamma='scale'
svm_model = SVC(kernel='rbf', random_state=42)

# Train the model using the training data
svm_model.fit(X_train, y_train)

print("SVM model (without Grid Search) trained successfully.")

### Evaluate SVM Model Performance

Now, let's evaluate the performance of the SVM model on the test set.

In [ ]:
# Make predictions on the test set
y_pred_svm = svm_model.predict(X_test)

# Calculate accuracy
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print(f"Accuracy of SVM Model (without Grid Search): {accuracy_svm:.4f}")

# Display classification report
print("\nClassification Report for SVM Model:")
print(classification_report(y_test, y_pred_svm))

# Display confusion matrix
print("\nConfusion Matrix for SVM Model:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_svm),
                     index=['Actual Negative', 'Actual Positive'],
                     columns=['Predicted Negative', 'Predicted Positive']))

## Exercise 5: SVM with Grid Search

### Instructions:
Implement an SVM classifier on the dataset with `GridSearchCV` to find the best combination of `C`, `kernel`, and `gamma` hyperparameters.

We will now use `GridSearchCV` to find the optimal hyperparameters for our SVM model. This involves defining a parameter grid for `C`, `kernel`, and `gamma`.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# Define the parameter grid for SVM
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
    'kernel': ['rbf', 'linear'] # Common kernels for SVM
}

# Initialize the SVM model
svm_model_gs = SVC(random_state=42)

# Initialize GridSearchCV for SVM
grid_search_svm = GridSearchCV(svm_model_gs, param_grid_svm, cv=5, scoring='accuracy', n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search_svm.fit(X_train, y_train)

print("GridSearchCV for SVM completed.")

### Best Parameters and Evaluation for SVM with Grid Search

Let's check the best parameters found and evaluate the performance of the best SVM estimator.

In [ ]:
# Print the best parameters found by GridSearchCV
print("Best parameters for SVM:", grid_search_svm.best_params_)

# Print the best cross-validation score
print("Best cross-validation accuracy:", grid_search_svm.best_score_)

# Get the best estimator (model) from GridSearchCV
best_svm_model = grid_search_svm.best_estimator_

# Make predictions on the test set using the best model
y_pred_best_svm = best_svm_model.predict(X_test)

# Calculate accuracy of the best model
accuracy_best_svm = accuracy_score(y_test, y_pred_best_svm)
print(f"\nAccuracy of Best SVM Model (with Grid Search): {accuracy_best_svm:.4f}")

# Display classification report for the best model
print("\nClassification Report for Best SVM Model:")
print(classification_report(y_test, y_pred_best_svm))

# Display confusion matrix for the best model
print("\nConfusion Matrix for Best SVM Model:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_best_svm),
                     index=['Actual Negative', 'Actual Positive'],
                     columns=['Predicted Negative', 'Predicted Positive']))

## Exercise 6: XGBoost without Grid Search

### Instructions:
Use the dataset to train an XGBoost classifier without hyperparameter tuning. Set the hyperparameters manually and justify your choices.

We'll use `XGBClassifier` with some reasonable default hyperparameters.

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize the XGBoost classifier with manually chosen hyperparameters
# n_estimators: number of boosting rounds (trees)
# learning_rate: step size shrinkage to prevent overfitting
# max_depth: maximum depth of a tree
# use_label_encoder=False and eval_metric='logloss' are for suppressing future warnings and setting a default metric

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic', # For binary classification
    n_estimators=100,            # Number of boosting rounds (trees)
    learning_rate=0.1,           # Step size shrinkage to prevent overfitting
    max_depth=3,                 # Maximum depth of a tree
    use_label_encoder=False,     # To suppress a deprecation warning
    eval_metric='logloss',       # Evaluation metric for binary classification
    random_state=42
)

# Train the model using the training data
xgb_model.fit(X_train, y_train)

print("XGBoost model (without Grid Search) trained successfully.")

### Evaluate XGBoost Model Performance

Now, let's evaluate the performance of the XGBoost model on the test set.

In [ ]:
# Make predictions on the test set
y_pred_xgb = xgb_model.predict(X_test)

# Calculate accuracy
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f"Accuracy of XGBoost Model (without Grid Search): {accuracy_xgb:.4f}")

# Display classification report
print("\nClassification Report for XGBoost Model:")
print(classification_report(y_test, y_pred_xgb))

# Display confusion matrix
print("\nConfusion Matrix for XGBoost Model:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_xgb),
                     index=['Actual Negative', 'Actual Positive'],
                     columns=['Predicted Negative', 'Predicted Positive']))

## Exercise 7: XGBoost with Grid Search

### Instructions:
Train an XGBoost classifier on the dataset using `GridSearchCV` to optimize hyperparameters such as `learning_rate`, `n_estimators`, `max_depth`, etc.

We will now use `GridSearchCV` to find the optimal hyperparameters for our XGBoost model. This will explore various combinations of `n_estimators`, `learning_rate`, and `max_depth`.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for XGBoost
param_grid_xgb = {
    'n_estimators': [50, 100, 200],      # Number of boosting rounds
    'learning_rate': [0.01, 0.1, 0.2],  # Step size shrinkage
    'max_depth': [3, 5, 7]              # Maximum depth of a tree
}

# Initialize the XGBoost model
xgb_model_gs = xgb.XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

# Initialize GridSearchCV for XGBoost
grid_search_xgb = GridSearchCV(xgb_model_gs, param_grid_xgb, cv=5, scoring='accuracy', n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search_xgb.fit(X_train, y_train)

print("GridSearchCV for XGBoost completed.")

### Best Parameters and Evaluation for XGBoost with Grid Search

Let's check the best parameters found and evaluate the performance of the best XGBoost estimator.

In [ ]:
# Print the best parameters found by GridSearchCV
print("Best parameters for XGBoost:", grid_search_xgb.best_params_)

# Print the best cross-validation score
print("Best cross-validation accuracy:", grid_search_xgb.best_score_)

# Get the best estimator (model) from GridSearchCV
best_xgb_model = grid_search_xgb.best_estimator_

# Make predictions on the test set using the best model
y_pred_best_xgb = best_xgb_model.predict(X_test)

# Calculate accuracy of the best model
accuracy_best_xgb = accuracy_score(y_test, y_pred_best_xgb)
print(f"\nAccuracy of Best XGBoost Model (with Grid Search): {accuracy_best_xgb:.4f}")

# Display classification report for the best model
print("\nClassification Report for Best XGBoost Model:")
print(classification_report(y_test, y_pred_best_xgb))

# Display confusion matrix for the best model
print("\nConfusion Matrix for Best XGBoost Model:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_best_xgb),
                     index=['Actual Negative', 'Actual Positive'],
                     columns=['Predicted Negative', 'Predicted Positive']))